## Ziel
<div class="alert alert-info"> VAE Imputation: Evaluation<br><br>
Bewertung der Imputationsgüte. 
Es wird verglichen, wie gut die vom VAE rekonstruierten Werte ("Imputed") mit den tatsächlichen Werten ("Original") übereinstimmen.

## Metriken
- **RMSE**: Root Mean Squared Error (Niedriger ist besser)
- **Verteilungs-Check**: Visueller Vergleich der Dichtefunktionen (KDE).

## Output
- Grafiken zur Imputationsqualität pro Feature.
- Zusammenfassung der Fehlerstatistik.
</div>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")

In [ ]:
# ------------------------- Pfad Definition -------------------------
base_dir = Path.cwd()
models_root = base_dir.parent / "4.1_VAE_Imputation" / "Models"

# ------------------------- Warten, bis der Ordner existiert -------------------------
import time
while not models_root.exists():
    time.sleep(1)


# ------------------------- neusten Modell Ordner finden -------------------------
latest_model_folder = None
while latest_model_folder is None:
    timestamp_folders = [f for f in models_root.iterdir() if f.is_dir()]
    if timestamp_folders:
        latest_model_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)
    else:
        time.sleep(2) # Warten, gab nämlich schon Probleme
        
print(f"Monitoring models from: {latest_model_folder.name}")


# ------------------------- Ergebnis Pfad setzen -------------------------
results_root = base_dir.parent / "4.2_Inference" / "Imputation_Results" / latest_model_folder.name
print(f"Monitoring results in: {results_root.name}")


# ------------------------- Seperater Ordner für Evaluation -------------------------
eval_output_dir = base_dir / "Evaluation_Results" / latest_model_folder.name
eval_output_dir.mkdir(parents=True, exist_ok=True)
print(f"Saving Evaluation to: {eval_output_dir}")


# ------------------------- Warten, bis der Ordner existiert -------------------------
while not results_root.exists():
    time.sleep(2)


# ------------------------- Evaluation Monitor Loop -------------------------
import time

processed_results = set()
all_metrics = []

print("Starte Evaluation Monitoring")

while True:
    result_files = sorted(list(results_root.glob("Imputation_Results_*.csv")))
    new_files = [f for f in result_files if f.name not in processed_results]
    
    if not new_files:
        if (results_root / "DONE_INFERENCE").exists():
            print("\nDONE_INFERENCE erkannt. Beende Evaluation.")
            break
        else:
            print(".", end="", flush=True)
            time.sleep(5)
            continue
            
    for res_file in new_files:
        processed_results.add(res_file.name)
        
        try:
            df_res = pd.read_csv(res_file)
            run_id = df_res['Run_ID'].iloc[0] if 'Run_ID' in df_res.columns else res_file.stem
            features_list_str = df_res['Features_List'].iloc[0] if 'Features_List' in df_res.columns else "Unknown"
            

            # ------------------------- Einfache Metriken berichtigen (RMSE, R2 per Feature) -------------------------
            for feature, group in df_res.groupby('Feature'):
                y_true = group['Original']
                y_pred = group['Imputed']
                
                mse = ((y_true - y_pred)**2).mean()
                rmse = np.sqrt(mse)
                
                # ------------------------- R2 Wert -------------------------
                ss_res = ((y_true - y_pred)**2).sum()
                ss_tot = ((y_true - y_true.mean())**2).sum()
                r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
                
                all_metrics.append({
                    'Run_ID': run_id,
                    'Feature': feature,
                    'Features_List': features_list_str,
                    'RMSE': rmse,
                    'R2': r2
                })
            
            print(f"Evaluated {res_file.name}")


            # ------------------------- PDF Bericht je Run -------------------------
            try:
                from matplotlib.backends.backend_pdf import PdfPages
                run_report_path = eval_output_dir / f"Analysis_{run_id}.pdf"
                with PdfPages(run_report_path) as pdf:

                    # ------------------------- Seite 1: Metriken -------------------------
                    fig, ax = plt.subplots(figsize=(11.69, 8.27))
                    ax.axis('off')
                    ax.text(0.5, 0.95, f"Analysis Report - {run_id}", ha='center', fontsize=24, weight='bold')
                    ax.text(0.5, 0.90, f"Date: {time.strftime('%Y-%m-%d %H:%M')}", ha='center', fontsize=12)
                    
                    run_metrics = [m for m in all_metrics if m['Run_ID'] == run_id]
                    if run_metrics:
                       df_run_t = pd.DataFrame(run_metrics)
                       cols_s = ['Feature', 'R2', 'RMSE']
                       if all(c in df_run_t.columns for c in cols_s):
                           df_s = df_run_t[cols_s].copy()
                           df_s['R2'] = df_s['R2'].round(4)
                           df_s['RMSE'] = df_s['RMSE'].round(4)
                           tbl_d = [df_s.columns.values.tolist()] + df_s.values.tolist()
                           tbl = ax.table(cellText=tbl_d, colLabels=df_s.columns, loc='center', cellLoc='center', bbox=[0.1, 0.1, 0.8, 0.7])
                           tbl.auto_set_font_size(False); tbl.set_fontsize(10)
                           ax.text(0.5, 0.82, 'Metrics', ha='center', fontsize=16)
                    pdf.savefig(fig); plt.close()
                    
                    # ------------------------- Seite 2: Scatter Plots Original vs. Imputed -------------------------
                    features_p = df_res['Feature'].unique()
                    cols = 3
                    rows = (len(features_p) + cols - 1) // cols
                    fig, axes = plt.subplots(rows, cols, figsize=(11.69, 4*rows))
                    fig.suptitle(f"{run_id} - Original vs Imputed", fontsize=16, y=1.02 if rows > 1 else 1.0)
                    
                    if rows == 1 and cols == 1:
                        axes = [axes]
                    else:
                        axes = axes.flatten()
                    
                    for i, feat in enumerate(features_p):
                        ax_p = axes[i]
                        sub = df_res[df_res['Feature'] == feat]

                        # ------------------------- Scatter -------------------------
                        sns.scatterplot(data=sub, x='Original', y='Imputed', ax=ax_p, alpha=0.3, s=10)
                        
                        # ------------------------- Ideal-Linie (Original = Imputed) -------------------------
                        if not sub.empty:
                             min_val = min(sub['Original'].min(), sub['Imputed'].min())
                             max_val = max(sub['Original'].max(), sub['Imputed'].max())
                             ax_p.plot([min_val, max_val], [min_val, max_val], 'r--', lw=1.5)
                        
                        # ------------------------- Metriken -------------------------
                        r2_val = 0
                        if 'df_run_t' in locals():
                             r2_row = df_run_t[df_run_t['Feature'] == feat]
                             if not r2_row.empty: r2_val = r2_row['R2'].iloc[0]
                        
                        ax_p.set_title(f"{feat} (R2={r2_val:.2f})")
                    
                    # ------------------------- Achsen verbergen -------------------------
                    for j in range(len(features_p), len(axes)):
                        axes[j].axis('off')
                    
                    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
                    pdf.savefig(fig)
                    plt.close()

                print(f"--> PDF Report for {run_id} created.")
            except Exception as e_pdf: print(f"Error creating PDF: {e_pdf}")

            
        except Exception as e:
            print(f"Error evaluating {res_file.name}: {e}")


Monitoring models from: 2026-01-09_20-14-13
Monitoring results in: 2026-01-09_20-14-13
Saving Evaluation to: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\4_Imputation\4.3_Evaluation\Evaluation_Results\2026-01-09_20-14-13
Starte Evaluation Monitoring...
.

.

.

.

.

...

Evaluated Imputation_Results_Run_029.csv
.

Evaluated Imputation_Results_Run_030.csv

DONE_INFERENCE erkannt. Beende Evaluation.

Zusammenfassung gespeichert: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\4_Imputation\4.3_Evaluation\Evaluation_Results\2026-01-09_20-14-13\Evaluation_Summary.csv
Run_ID
Run_006    0.550098
Run_014    0.547496
Run_007    0.524647
Run_021    0.494329
Run_009    0.488426
Name: R2, dtype: float64


Generating PDF Report: Evaluation_Report.pdf...


C:\Users\lucca\AppData\Local\Temp\ipykernel_37848\1607162709.py:138: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45)


PDF Saved.


In [ ]:
# ------------------------- Finalen Bericht fertigstellen -------------------------
if all_metrics:
    df_metrics = pd.DataFrame(all_metrics)
    summary_path = eval_output_dir / "Evaluation_Summary.csv"
    df_metrics.to_csv(summary_path, index=False)
    print(f"\nZusammenfassung gespeichert: {summary_path}")
    
    # ------------------------- Top 5 anzeugen via .head -------------------------
    print(df_metrics.groupby('Run_ID')['R2'].mean().sort_values(ascending=False).head())


    # ------------------------- PDF Report erstellen -------------------------
    from matplotlib.backends.backend_pdf import PdfPages
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    pdf_path = eval_output_dir / "Evaluation_Report.pdf"
    print(f"Generating PDF Report: {pdf_path.name}...")
    
    with PdfPages(pdf_path) as pdf:

        # ------------------------- Seite 1: Titel und bestes Modell -------------------------
        fig, ax = plt.subplots(figsize=(11.69, 8.27)) # A4 Landscape
        ax.axis('off')
        ax.text(0.5, 0.95, "VAE Imputation - Evaluation Report", ha='center', fontsize=24, weight='bold')
        ax.text(0.5, 0.90, f"Date: {time.strftime('%Y-%m-%d %H:%M')}", ha='center', fontsize=12)
        
        # ------------------------- Bestes Modell -------------------------
        agg_funcs = {'R2': 'mean', 'RMSE': 'mean', 'Features_List': 'first'}
        top_n = df_metrics.groupby('Run_ID').agg(agg_funcs).sort_values('R2', ascending=False).head(10).reset_index()
        top_n['R2'] = top_n['R2'].round(4)
        top_n['RMSE'] = top_n['RMSE'].round(4)
        
        table_data = [top_n.columns.values.tolist()] + top_n.values.tolist()
        table = ax.table(cellText=table_data, colLabels=top_n.columns, loc='center', cellLoc='center', bbox=[0.1, 0.4, 0.8, 0.4])
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 1.2)
        ax.text(0.5, 0.82, "Top 10 Models (by Mean R²)", ha='center', fontsize=16)
        
        pdf.savefig(fig)
        plt.close()
        
        # ------------------------- Seite 2: Performanz-Bericht -------------------------
        fig, ax = plt.subplots(figsize=(11.69, 8.27))
        sns.boxplot(data=df_metrics, x='Feature', y='R2', ax=ax)
        ax.set_title("R² Score Distribution per Feature across all Runs", fontsize=16)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
        ax.set_ylim(-0.1, 1.1)
        pdf.savefig(fig)
        plt.close()
        
        # ------------------------- Seite 3: Details zum besten Modell -------------------------
        best_run_id = top_n.iloc[0]['Run_ID']
        best_run_file = results_root / f"Imputation_Results_{best_run_id}.csv"
        
        if best_run_file.exists():
            df_best = pd.read_csv(best_run_file)
            features = df_best['Feature'].unique()
            
# ------------------------- Scatter Plots für bestes Modell -------------------------
            cols = 3
            rows = (len(features) + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(11.69, 4*rows))
            fig.suptitle(f"Best Model: {best_run_id} - Original vs Imputed", fontsize=16)
            axes = axes.flatten()
            
            for i, feat in enumerate(features):
                ax = axes[i]
                subset = df_best[df_best['Feature'] == feat]
                
                # ------------------------- Scatter Plot -------------------------
                sns.scatterplot(data=subset, x='Original', y='Imputed', ax=ax, alpha=0.3, s=10)
                
                # ------------------------- Ideallinie (Original vs. Imputation) -------------------------
                min_val = min(subset['Original'].min(), subset['Imputed'].min())
                max_val = max(subset['Original'].max(), subset['Imputed'].max())
                ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=1.5)
                
                
                # ------------------------- Metriken -------------------------
                r2_vals = df_metrics[(df_metrics['Run_ID'] == best_run_id) & (df_metrics['Feature'] == feat)]['R2'].values
                r2_val = r2_vals[0] if len(r2_vals) > 0 else 0.0

                ax.set_title(f"{feat} (R²: {r2_val:.2f})")
                
            for j in range(i+1, len(axes)): axes[j].axis('off')
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            pdf.savefig(fig)
            plt.close()
            
    print(f"PDF Saved.")